In [2]:
%pwd

'e:\\ml\\DataScience_Project1\\research'

In [3]:
import os
os.chdir("../")
%pwd

'e:\\ml\\DataScience_Project1'

In [4]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class DataTransformationConfig:
    root_dir: Path
    transformed_train_file: Path
    transformed_test_file: Path
    data_path: Path

In [5]:
import yaml
from box import ConfigBox
from pathlib import Path


def read_yaml(path_to_yaml: Path) -> ConfigBox:
    with open(path_to_yaml) as yaml_file:
        content = yaml.safe_load(yaml_file)
        return ConfigBox(content)

In [6]:
import logging
import os

LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)

LOG_FILE = os.path.join(LOG_DIR, "running.log")

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format='[%(asctime)s] %(lineno)d %(name)s - %(levelname)s - %(message)s',
)

logging.getLogger().addHandler(logging.StreamHandler())

In [7]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories
from src.datascience.entity.config_entity import DataTransformationConfig
from pathlib import Path


class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])
        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            transformed_train_file=config.transformed_train_file,
            transformed_test_file=config.transformed_test_file
        )
        return data_transformation_config
            


        

In [9]:
import os
from src.datascience.constants import *
from src.datascience.utils.logger import logging
from sklearn.model_selection import train_test_split
from src.datascience.utils.common import read_yaml, create_directories
import pandas as pd



In [10]:
from pathlib import Path
from src.datascience.utils.logger import logging
import pandas as pd
from sklearn.model_selection import train_test_split


class DataTransformation:
    def __init__(self, config):
        self.config = config

    def train_test_split(self):  # ✅ fixed name
        # create folder
        Path(self.config.root_dir).mkdir(parents=True, exist_ok=True)

        logging.info("Reading the data from the data path")

        df = pd.read_csv(self.config.data_path, sep=";")  # ✅ fixed

        logging.info("Splitting the data into train and test sets")

        train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

        logging.info("Saving transformed data")

        train_df.to_csv(self.config.transformed_train_file, index=False)
        test_df.to_csv(self.config.transformed_test_file, index=False)

        logging.info("Data transformation completed")

        logging.info(f"Train file: {self.config.transformed_train_file}")
        logging.info(f"Test file: {self.config.transformed_test_file}")

        logging.info(f"Train shape: {train_df.shape}")
        logging.info(f"Test shape: {test_df.shape}")

In [12]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.train_test_split()
except Exception as e:
    raise e

YAML file loaded successfully from: config\config.yaml


[2026-03-19 17:29:54,499] 29 root - INFO - YAML file loaded successfully from: config\config.yaml


YAML file loaded successfully from: params.yaml


[2026-03-19 17:29:54,503] 29 root - INFO - YAML file loaded successfully from: params.yaml


YAML file loaded successfully from: schema.yaml


...

[2026-03-19 17:29:54,574] 31 root - INFO - Test file: artifacts/data_transformation/transformed_test.csv


Train shape: (1279, 12)


[2026-03-19 17:29:54,574] 33 root - INFO - Train shape: (1279, 12)


Test shape: (320, 12)


[2026-03-19 17:29:54,576] 34 root - INFO - Test shape: (320, 12)
